#### **SOLUCION BASE ALIXPARTNERS**

In [26]:
import pandas as pd 
import matplotlib.pyplot as plt
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")

In [27]:
caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")

cajas = [
    Caja(
        caja_id = row["caja_tipo_id"],
        dim_interior_ancho = row["caja_interior_ancho"],
        dim_interior_largo = row["caja_interior_largo"],
        dim_interior_alto = row["caja_interior_alto"],
        costo_unitario = row['costo_unitario_base']
    )
    for _, row in caja_compras_merge.iterrows()
]

cajas[:5]

[<Caja 02cf77de65b70dd77905e2e33d78478f | Int: 296.0 x 395.0 x 291.0mm | Compra Total: 0>,
 <Caja 082c1cdb42b1abd201403ca33ca11ef0 | Int: 248.0 x 383.0 x 189.0mm | Compra Total: 0>,
 <Caja 0835ff365412a67b720a19713ec250f3 | Int: 286.0 x 386.0 x 278.0mm | Compra Total: 0>,
 <Caja 0b72571a5bb7429ce7de424547e8d27d | Int: 286.0 x 386.0 x 174.0mm | Compra Total: 0>,
 <Caja 10c5f9edbe2c87186bcdeb991fe8d902 | Int: 252.0 x 380.0 x 228.0mm | Compra Total: 0>]

In [28]:
prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")

productos = [
    Producto(
        codigo_producto = row['codigo_producto'],
        cantidad_paquetes = row['cantidad_paquetes'],
        peso_paquete = row['peso_neto_paquete'],
        demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
        demanda_curitiba = row['volumen_producto_planta_curitiba'],
        demanda_santiago = row['volumen_producto_planta_santiago'],
        demanda_monterrey = row['volumen_producto_planta_monterrey'],
        demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
        dim_producto_ancho = row['dim_producto_ancho'], 
        dim_producto_largo = row['dim_producto_largo'],
        dim_producto_alto = row['dim_producto_alto']
    )
    for _, row in prod_op_merge.iterrows()
]

productos[:5]

[<Producto BR0001 | Dim Prod: 286.0 x 386.0 x 303.0mm | Demanda Total: 1546613>,
 <Producto BR0002 | Dim Prod: 296.0 x 395.0 x 260.0mm | Demanda Total: 139211>,
 <Producto BR0003 | Dim Prod: 288.0 x 388.0 x 164.0mm | Demanda Total: 172506>,
 <Producto BR0004 | Dim Prod: 296.0 x 395.0 x 224.0mm | Demanda Total: 271715>,
 <Producto BR0005 | Dim Prod: 286.0 x 386.0 x 253.0mm | Demanda Total: 7586>]

In [29]:
def buscar_caja_por_id(id):
    caja_a_buscar = None
    for caja in cajas:
        if caja.caja_id == id:
            caja_a_buscar = caja
    return caja_a_buscar

prod_cajas_merge = catalogo_productos.merge(especificaciones_cajas,
                                            on="caja_tipo_id", how="left")

solucion = Solucion()

for producto in productos:    
    fila = prod_cajas_merge[
        prod_cajas_merge['codigo_producto'] == producto.codigo_producto
    ]
    
    caja_id = fila['caja_tipo_id'].iloc[0]
    caja = buscar_caja_por_id(caja_id)
    grosor_caja = fila['caja_grosor_mm'].iloc[0] 
    
    caja.elegir_grosor(grosor_caja)
    asignacion = Asignacion(producto, caja)
    solucion.agregar_asignacion(asignacion, descuentos=True)
    
solucion.exportar_submmit(nombre_csv="0")
solucion.resumen_por_asignacion()

,codigo_producto,demanda_total,caja_id,utilizacion_pallet,utilizacion_caja,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,a23eeef1c0cd8cfd40b780d65382034a,0.835440,1.0,720369.520,38667,5800050,6520419.520
1,BR0002,139211,3256a5fed07b421a92abee44a3d93243,0.333631,1.0,66821.280,7734,1160100,1226921.280
2,BR0003,172506,4f08f853cf7f9be28571df66a50bc322,0.935576,1.0,89703.120,2157,323550,413253.120
3,BR0004,271715,c96df5913565ba35f3e02e744c9a6dde,0.336439,1.0,130423.200,12939,1940850,2071273.200
4,BR0005,7586,eb4895380ccaa410da0fefb6f12b197b,0.841453,1.0,3944.720,159,23850,27794.720
...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,563a3fa61d441769163b35905be8987d,0.786677,1.0,1974.115,50,7500,9474.115
423,BR0418,3942,563a3fa61d441769163b35905be8987d,0.786677,1.0,2818.530,71,10650,13468.530
424,BR0419,43300,3cdb44bb6e06ba2151d665a0c599b894,0.373775,1.0,25980.000,1444,216600,242580.000
425,BR0420,205852,a130566667d4be7bb38053adaefea0af,0.804587,1.0,87911.460,3217,482550,570461.460
